# Database3 Identity Overlap Between Train and Test

This notebook inspects how much identity-like overlap exists between the challenge training set and the challenge test set for `database3`.

In `database3`, paths usually look like:

```text
database3/database3/m.0109kg/0-FaceId-0_align.webp
```

The `m.<id>` folder is treated as an identity-like group. This matters for validation strategy because identities can appear in both challenge train and challenge test.

After running the notebook on the current CSVs, the main result is very clear:

- `database3` contains **96,023 / 100,000** train rows and **28,435 / 29,980** test rows.
- The challenge test set has **10,795** unique `database3` person IDs.
- **10,497** of those test IDs also appear in train, which means **97.24%** of database3 test IDs are seen during training.
- At the row level, **28,036 / 28,435** database3 test rows have a person ID seen in train, which is **98.60%**.

This notebook reads only `train.csv` and `test_students.csv`. It does not load image pixels or run model inference.


## 1. Setup

We reuse the project metadata parser so the notebook uses the same `database`, `group_id`, and `face_id` logic as training, validation, and diagnostics.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))

from face_occlusion.data.metadata import add_path_metadata  # noqa: E402

DATA_ROOT = ROOT / "data"
TRAIN_CSV = DATA_ROOT / "raw" / "occlusion_datasets" / "train.csv"
TEST_CSV = DATA_ROOT / "raw" / "occlusion_datasets" / "test_students.csv"

FILENAME_COL = "filename"
TARGET_COL = "FaceOcclusion"
GENDER_COL = "gender"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

TRAIN_CSV, TEST_CSV

## 2. Load CSVs and Parse Database3 Person IDs

For `database3`, the project parser returns `group_id` values like `database3/m.0109kg`. We remove the `database3/` prefix to get the notebook-level `person_id`.

In [ ]:
def database3_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Keep database3 rows and derive the m.<id> person_id."""
    out = add_path_metadata(df, filename_col=FILENAME_COL)
    out = out[out["database"] == "database3"].copy()

    # Expected database3 group_id format: database3/m.<id>
    is_database3_person = out["group_id"].astype(str).str.match(r"^database3/m\.[^/]+$")
    out["person_id"] = out["group_id"].astype(str).str.replace("database3/", "", regex=False)
    out.loc[~is_database3_person, "person_id"] = pd.NA
    return out


train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

train_db3 = database3_rows(train)
test_db3 = database3_rows(test)

print(f"train rows: {len(train):,}")
print(f"test rows:  {len(test):,}")
print(f"database3 train rows: {len(train_db3):,}")
print(f"database3 test rows:  {len(test_db3):,}")

## 3. How Dominant Is Database3?

`database3` is the main source in both challenge train and challenge test:

- train: **96,023 rows**, or **96.02%** of all train rows;
- test: **28,435 rows**, or **94.85%** of all test rows.

Because `database3` dominates both splits, its identity overlap has a large effect on how we should interpret validation scores.


In [ ]:
database3_share = pd.DataFrame(
    {
        "split": ["train", "test"],
        "database3_rows": [len(train_db3), len(test_db3)],
        "other_rows": [len(train) - len(train_db3), len(test) - len(test_db3)],
    }
)
database3_share["database3_percentage"] = database3_share["database3_rows"] / (
    database3_share["database3_rows"] + database3_share["other_rows"]
)

display(database3_share)

plot_df = database3_share.melt(
    id_vars="split",
    value_vars=["database3_rows", "other_rows"],
    var_name="source",
    value_name="rows",
)

ax = sns.barplot(data=plot_df, x="split", y="rows", hue="source")
ax.set_title("Database3 dominates both train and test")
ax.set_xlabel("")
ax.set_ylabel("rows")
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
ax.bar_label(ax.containers[1], fmt="%d", padding=3)
plt.show()

## 4. Unique Person IDs and Overlap

We now compare the unique `m.<id>` identities in challenge train and challenge test.

The overlap is very high:

- train has **15,865** unique database3 person IDs;
- test has **10,795** unique database3 person IDs;
- **10,497** IDs appear in both train and test;
- only **298** database3 test IDs are test-only.

The most important number is the test seen ID ratio: **97.24%** of database3 test IDs are already present in train.


In [ ]:
train_id_counts = (
    train_db3.dropna(subset=["person_id"])
    .groupby("person_id")
    .size()
    .rename("train_count")
    .reset_index()
)
test_id_counts = (
    test_db3.dropna(subset=["person_id"])
    .groupby("person_id")
    .size()
    .rename("test_count")
    .reset_index()
)

train_ids = set(train_id_counts["person_id"])
test_ids = set(test_id_counts["person_id"])
shared_ids = train_ids & test_ids

train_id_counts["is_in_test"] = train_id_counts["person_id"].isin(test_ids)
test_id_counts["is_in_train"] = test_id_counts["person_id"].isin(train_ids)

id_summary = pd.DataFrame(
    [
        ["unique train IDs", len(train_ids)],
        ["unique test IDs", len(test_ids)],
        ["shared IDs", len(shared_ids)],
        ["train-only IDs", len(train_ids - test_ids)],
        ["test-only IDs", len(test_ids - train_ids)],
        ["test seen ID ratio", len(shared_ids) / len(test_ids)],
        ["train IDs also in test", len(shared_ids) / len(train_ids)],
    ],
    columns=["metric", "value"],
)

display(id_summary)

In [ ]:
overlap_bars = pd.DataFrame(
    {
        "population": ["test IDs", "database3 test rows"],
        "seen_in_train": [
            len(shared_ids),
            test_db3["person_id"].isin(train_ids).sum(),
        ],
        "unseen_in_train": [
            len(test_ids - train_ids),
            (~test_db3["person_id"].isin(train_ids)).sum(),
        ],
    }
)
overlap_bars["seen_ratio"] = overlap_bars["seen_in_train"] / (
    overlap_bars["seen_in_train"] + overlap_bars["unseen_in_train"]
)

display(overlap_bars)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(overlap_bars["population"], overlap_bars["seen_ratio"], label="seen in train")
ax.barh(
    overlap_bars["population"],
    1 - overlap_bars["seen_ratio"],
    left=overlap_bars["seen_ratio"],
    label="unseen in train",
)
ax.set_xlim(0, 1)
ax.set_xlabel("share")
ax.set_title("Most database3 test identities are already present in training")
ax.legend(loc="lower right")

for row_idx, row in overlap_bars.iterrows():
    ax.text(row["seen_ratio"] / 2, row_idx, f"{row['seen_ratio']:.1%}", va="center")

plt.show()

## 5. Image Count per Person ID

Overlap by unique ID is important, but row-level weight matters too. In this data, the shared identities also account for most database3 test images:

- **98.60%** of database3 test rows have a person ID seen in train;
- **75.35%** of database3 train rows have a person ID that also appears in test.

Typical database3 IDs have several images. The median is **5** train images per ID and **2** test images per ID. For shared IDs, the median is **6** train images and **2** test images.


In [ ]:
def count_stats(series: pd.Series) -> pd.Series:
    return pd.Series(
        {
            "min": series.min(),
            "median": series.median(),
            "mean": series.mean(),
            "max": series.max(),
            "p90": series.quantile(0.90),
            "p95": series.quantile(0.95),
            "p99": series.quantile(0.99),
        }
    )


shared_train_counts = train_id_counts[train_id_counts["person_id"].isin(shared_ids)]
shared_test_counts = test_id_counts[test_id_counts["person_id"].isin(shared_ids)]

image_count_summary = pd.DataFrame(
    {
        "train images per ID": count_stats(train_id_counts["train_count"]),
        "test images per ID": count_stats(test_id_counts["test_count"]),
        "shared IDs: train images": count_stats(shared_train_counts["train_count"]),
        "shared IDs: test images": count_stats(shared_test_counts["test_count"]),
    }
).T

display(image_count_summary)

In [ ]:
hist_df = pd.concat(
    [
        train_id_counts[["train_count"]]
        .rename(columns={"train_count": "images"})
        .assign(split="train"),
        test_id_counts[["test_count"]]
        .rename(columns={"test_count": "images"})
        .assign(split="test"),
    ],
    ignore_index=True,
)

ax = sns.histplot(data=hist_df, x="images", hue="split", discrete=True, multiple="dodge")
ax.set_yscale("log")
ax.set_title("Images per database3 person ID")
ax.set_xlabel("images for one m.<id>")
ax.set_ylabel("number of IDs, log scale")
plt.show()

## 6. Most Repeated Shared IDs

These IDs have the largest combined train + test footprint. They are useful to inspect manually if a model seems to overfit identity-specific cues.

The most repeated shared ID in this run is `m.0134wr`, with **29** train images and **11** test images, for **40** images total. Several other shared IDs have around **27 to 35** total images.


In [ ]:
shared_counts = (
    train_id_counts[train_id_counts["person_id"].isin(shared_ids)]
    .merge(test_id_counts, on="person_id", how="inner")
    .assign(total_count=lambda df: df["train_count"] + df["test_count"])
    .sort_values("total_count", ascending=False)
)

top_shared = shared_counts.head(20).copy()
display(top_shared)

plot_top_shared = top_shared.sort_values("total_count")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_top_shared["person_id"], plot_top_shared["train_count"], label="train")
ax.barh(
    plot_top_shared["person_id"],
    plot_top_shared["test_count"],
    left=plot_top_shared["train_count"],
    label="test",
)
ax.set_title("Top shared database3 IDs by total image count")
ax.set_xlabel("images")
ax.legend()
plt.show()

## 7. Target Distribution: Shared vs Non-Shared Train IDs

The test CSV has no target, so we compare train rows whose IDs appear in test against train rows whose IDs do not appear in test.

The two groups are not identical:

- train rows whose IDs also appear in test have mean target **0.0927** and median target **0.0619**;
- train-only IDs have mean target **0.0470** and median target **0.0262**.

So the identities that also appear in test have, on average, higher occlusion labels in the training data.


In [ ]:
train_db3 = train_db3.assign(
    identity_status=train_db3["person_id"]
    .isin(test_ids)
    .map({True: "ID also in test", False: "train-only ID"})
)

target_summary = (
    train_db3.groupby("identity_status")[TARGET_COL]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=train_db3, x="identity_status", y=TARGET_COL, ax=axes[0])
axes[0].set_title("Target distribution by identity overlap")
axes[0].set_xlabel("")
axes[0].set_ylabel("FaceOcclusion")

sns.histplot(
    data=train_db3,
    x=TARGET_COL,
    hue="identity_status",
    bins=50,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1],
)
axes[1].set_title("Shared-ID train rows have a higher target distribution")
axes[1].set_xlabel("FaceOcclusion")
plt.tight_layout()
plt.show()

## 8. Gender Distribution: Shared vs Non-Shared Train IDs

The confirmed dataset encoding is `female = 0`, `male = 1`. The test CSV does not provide gender, so this comparison is train-only.

The gender distribution also differs between shared and train-only IDs:

- shared-with-test train rows: **37.55% female**, **62.45% male**;
- train-only rows: **11.45% female**, **88.55% male**.

This means identity overlap is also linked with metadata distribution, not only with person ID reuse.


In [ ]:
gender_counts = (
    train_db3.groupby(["identity_status", GENDER_COL]).size().rename("count").reset_index()
)
gender_counts["gender_label"] = (
    gender_counts[GENDER_COL].astype(int).map({0: "female (0)", 1: "male (1)"})
)
gender_counts["percentage"] = gender_counts["count"] / gender_counts.groupby("identity_status")[
    "count"
].transform("sum")

display(gender_counts)

ax = sns.barplot(
    data=gender_counts,
    x="identity_status",
    y="percentage",
    hue="gender_label",
)
ax.set_title("Gender distribution differs between shared and train-only IDs")
ax.set_xlabel("")
ax.set_ylabel("within-group percentage")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()

## 9. FaceId Distribution

Most images use `FaceId-0`, but non-zero `FaceId` values exist. These values should not be model features, but they are useful diagnostics.

`FaceId-0` dominates both splits:

- train: **92,614 / 96,023** database3 rows;
- test: **27,487 / 28,435** database3 rows.

Non-zero `FaceId` values are rare but present, especially `FaceId-1` and `FaceId-2`. Keeping `face_id` in diagnostics helps detect whether these rare cases behave differently.


In [ ]:
face_id_df = pd.concat(
    [
        train_db3[["face_id"]].assign(split="train"),
        test_db3[["face_id"]].assign(split="test"),
    ],
    ignore_index=True,
)

face_counts = (
    face_id_df.groupby(["split", "face_id"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "count"], ascending=[True, False])
)

display(face_counts.groupby("split").head(10))

top_face_ids = face_counts.groupby("face_id")["count"].sum().nlargest(12).index
plot_face_counts = face_counts[face_counts["face_id"].isin(top_face_ids)]

ax = sns.barplot(data=plot_face_counts, x="face_id", y="count", hue="split")
ax.set_yscale("log")
ax.set_title("FaceId distribution, top values")
ax.set_xlabel("FaceId")
ax.set_ylabel("rows, log scale")
plt.show()

## 10. Final Interpretation

The final takeaway is static because the notebook has now been run on the current data and the measured overlap is stable for these CSVs.


### Main takeaway

Database3 has very high identity-like overlap between challenge train and challenge test.

Key results from this run:

- Database3 train rows: **96,023** out of **100,000** train rows, or **96.02%**.
- Database3 test rows: **28,435** out of **29,980** test rows, or **94.85%**.
- Unique database3 train IDs: **15,865**.
- Unique database3 test IDs: **10,795**.
- Shared train/test database3 IDs: **10,497**.
- Test-only database3 IDs: **298**.
- Test database3 IDs already seen in train: **97.24%**.
- Database3 test rows whose ID is already seen in train: **98.60%**.

This strongly suggests that the canonical `row_stratified` validation split is relevant for leaderboard-like model comparison, because the challenge test set itself contains many identities already present in training.

However, this does not make `group_stratified` unnecessary. The group-level split remains important because it tests whether the model generalizes to unseen identity-like groups instead of relying too much on identity-specific visual cues.

Recommended validation strategy:

1. Use `row_stratified` as the main comparison split for leaderboard-oriented experiments.
2. Use `group_stratified` as a robustness check for identity memorization.
3. Track diagnostics by `database`, `group_id`, `gender`, `face_id`, and occlusion bin when comparing models.

One extra caution: shared-with-test train rows have higher targets on average than train-only rows (**0.0927** vs **0.0470**) and a less male-heavy gender distribution. This means identity overlap is also tied to distribution differences, so validation results should be interpreted with metadata diagnostics, not only with one aggregate score.
